In [96]:
import pandas as pd
import plotly.express as px

In [97]:
df_contenedores = pd.read_csv(
    "../data/raw/sigab_contenedores.csv",
    sep="|",
    encoding="latin-1")
df_pqrs = pd.read_csv("../data/raw/sigab_pqrsxlocalidad.csv", 
                      sep="|",
                      encoding="latin-1")
df_puntos_criticos = pd.read_csv("../data/raw/sigab_puntoscriticos.csv", 
                                 sep="|", 
                                 encoding="latin-1")
df_id_localidades = pd.read_excel('../data/raw/nombres_localidades.xlsx')

## 1. Comprensión de las fuentes

##### 1. Dataset: sigab_contenedores.csv

Unidad de análisis: 

Cobertura: 

Variables relevantes: 

Calidad: 

Limitaciones: 

Posibles llaves de integración: 


##### 2. Dataset: sigab_pqrsxlocalidad.csv

Unidad de análisis: 
Cobertura: 
Variables relevantes: 
Calidad: 
Limitaciones: 
Posibles llaves de integración: 

##### 3. Dataset: sigab_puntoscriticos.csv

Unidad de análisis: 
Cobertura: 
Variables relevantes: 
Calidad: 
Limitaciones: 
Posibles llaves de integración: 

##### 4. Dataset: .csv

Unidad de análisis: 
Cobertura: 
Variables relevantes: 
Calidad: 
Limitaciones: 
Posibles llaves de integración: 

## 2. Preparación e integración de las fuentes

##### Dataset contenedores

In [98]:
df_contenedores.head()

,CONCESIONARIO,CODINTERNO,IDLOCALID,DIRECCION,CAPACIDAD,APROVECHAB,SERIAL,FECVIGDES,FECHAINSTA,LONGITUD,LATITUD,NOVUBICACI,FECHANOVED
0,Ciudad Límpia (ASE03),24-0849,8,PENDIENTE REPOSICION EN ESPERA DE LINEAMIENTOS...,2400,N,NaN,2025-11-01 00:00:00,NaN,-74.164473,4.607639,RE,2023-03-31
1,Ciudad Límpia (ASE03),24-0618,8,CL 57I SUR No 77I 71,2400,N,NaN,2025-11-01 00:00:00,NaN,-74.173674,4.604760,UC,2018-11-30
2,Ciudad Límpia (ASE03),24-0532,8,CL 58B SUR No 77J 21,2400,N,NaN,2025-11-01 00:00:00,NaN,-74.175146,4.605122,UC,2018-12-17
3,Ciudad Límpia (ASE03),24-0613,8,CL 58 SUR No 77H 63,2400,N,NaN,2025-11-01 00:00:00,NaN,-74.174162,4.604571,UC,2018-11-30
4,Ciudad Límpia (ASE03),24-0563,8,CL 57C SUR No 77H 26,2400,N,NaN,2025-11-01 00:00:00,NaN,-74.172719,4.604418,UC,2018-12-17


In [99]:
df_contenedores.info()

<class 'pandas.DataFrame'>
RangeIndex: 9865 entries, 0 to 9864
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   CONCESIONARIO  9865 non-null   str    
 1   CODINTERNO     9865 non-null   str    
 2   IDLOCALID      9865 non-null   int64  
 3   DIRECCION      9865 non-null   str    
 4   CAPACIDAD      9865 non-null   int64  
 5   APROVECHAB     9865 non-null   str    
 6   SERIAL         3910 non-null   str    
 7   FECVIGDES      9865 non-null   str    
 8   FECHAINSTA     0 non-null      float64
 9   LONGITUD       9865 non-null   float64
 10  LATITUD        9865 non-null   float64
 11  NOVUBICACI     9865 non-null   str    
 12  FECHANOVED     9865 non-null   str    
dtypes: float64(3), int64(2), str(8)
memory usage: 1.7 MB


In [100]:
columns_contenedores = ['CONCESIONARIO', 'IDLOCALID','CODINTERNO', 'NOVUBICACI']
df_contenedores = df_contenedores[columns_contenedores]
df_contenedores


,CONCESIONARIO,IDLOCALID,CODINTERNO,NOVUBICACI
0,Ciudad Límpia (ASE03),8,24-0849,RE
1,Ciudad Límpia (ASE03),8,24-0618,UC
2,Ciudad Límpia (ASE03),8,24-0532,UC
3,Ciudad Límpia (ASE03),8,24-0613,UC
4,Ciudad Límpia (ASE03),8,24-0563,UC
...,...,...,...,...
9860,PromoAmbiental (ASE01),5,AP201P035,ET
9861,PromoAmbiental (ASE01),5,AP201P036,ET
9862,PromoAmbiental (ASE01),5,AP201P037,ET
9863,PromoAmbiental (ASE01),5,AP201P038,ET


In [101]:
df_contenedores_limpio = df_contenedores[(df_contenedores['NOVUBICACI'] == 'UC') | (df_contenedores['NOVUBICACI'] == 'ET')]

In [102]:
contenedores = df_contenedores_limpio.groupby(['CONCESIONARIO', 'IDLOCALID'], as_index=False).agg(
    num_contenedores=('CODINTERNO', 'count'))
contenedores

,CONCESIONARIO,IDLOCALID,num_contenedores
0,Bogotá Límpia (ASE04),10,2205
1,Bogotá Límpia (ASE04),12,351
2,Ciudad Límpia (ASE03),8,1195
3,Ciudad Límpia (ASE03),9,583
4,Lime (ASE02),6,226
5,Lime (ASE02),7,362
6,Lime (ASE02),13,173
7,Lime (ASE02),14,97
8,Lime (ASE02),15,43
9,Lime (ASE02),16,267


##### Dataset puntos criticos

In [103]:
df_puntos_criticos.head()

,CONCESIONARIO,IDLOCALID,DIRECCION,FECVIGDES,CODIGOINTE,IDFRECUE,FRECUENCIA,EXTENSION,OBSERVACION,LONGITUD,LATITUD
0,Ciudad Límpia (ASE03),8,CALLE 42 SUR CON CARRERA 72K,2025-12-01 00:00:00,PC08002,11,Viernes,0.0,NaN,-74.150190,4.606172
1,Ciudad Límpia (ASE03),8,AV. CIUDAD DE CALI CON CALLE 15A,2025-12-01 00:00:00,PC08003,2,Martes - Jueves - Sabado,0.0,NaN,-74.138619,4.656549
2,Ciudad Límpia (ASE03),8,CALLE 38 SUR ENTRE CARRERAS 86 A LA 89,2025-12-01 00:00:00,PC08004,10,Jueves,0.0,NaN,-74.166914,4.639724
3,Ciudad Límpia (ASE03),8,CALLE 55 SUR CON CARRERA 80A A LA CARRERA 80G,2025-12-01 00:00:00,PC08005,1,Lunes - Miercoles - Viernes,0.0,NaN,-74.175104,4.614885
4,Ciudad Límpia (ASE03),8,CARRERA 77H CON CALLE 49 SUR,2025-12-01 00:00:00,PC08006,2,Martes - Jueves - Sabado,0.0,NaN,-74.166138,4.605319


In [104]:
df_puntos_criticos.info()

<class 'pandas.DataFrame'>
RangeIndex: 711 entries, 0 to 710
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   CONCESIONARIO  711 non-null    str    
 1   IDLOCALID      711 non-null    int64  
 2   DIRECCION      711 non-null    str    
 3   FECVIGDES      711 non-null    str    
 4   CODIGOINTE     711 non-null    str    
 5   IDFRECUE       711 non-null    int64  
 6   FRECUENCIA     711 non-null    str    
 7   EXTENSION      711 non-null    float64
 8   OBSERVACION    606 non-null    str    
 9   LONGITUD       711 non-null    float64
 10  LATITUD        711 non-null    float64
dtypes: float64(3), int64(2), str(6)
memory usage: 119.5 KB


In [108]:
columns_puntos_criticos = ['CONCESIONARIO', 'IDLOCALID', 'CODIGOINTE', 'FRECUENCIA']
df_puntos_criticos_filtrado = df_puntos_criticos[columns_puntos_criticos]
df_puntos_criticos_filtrado['DIAS'] = df_puntos_criticos['FRECUENCIA'].str.count('-').add(1).fillna(0)
df_puntos_criticos_filtrado


,CONCESIONARIO,IDLOCALID,CODIGOINTE,FRECUENCIA,DIAS
0,Ciudad Límpia (ASE03),8,PC08002,Viernes,1
1,Ciudad Límpia (ASE03),8,PC08003,Martes - Jueves - Sabado,3
2,Ciudad Límpia (ASE03),8,PC08004,Jueves,1
3,Ciudad Límpia (ASE03),8,PC08005,Lunes - Miercoles - Viernes,3
4,Ciudad Límpia (ASE03),8,PC08006,Martes - Jueves - Sabado,3
...,...,...,...,...,...
706,PromoAmbiental (ASE01),5,0509,Martes - Viernes,2
707,PromoAmbiental (ASE01),5,0510,Martes - Viernes,2
708,PromoAmbiental (ASE01),17,1701,Lunes - Jueves,2
709,PromoAmbiental (ASE01),17,1702,Lunes - Jueves,2


In [114]:
df_puntos_criticos_agrupado = df_puntos_criticos.groupby(['CONCESIONARIO','IDLOCALID'], as_index=False).agg(NUM_PUNTOS_CRITICOS = ('CODIGOINTE', 'count'), TOTAL_OCURRENCIAS_SEMANALES = ('DIAS', 'sum'))
df_puntos_criticos_agrupado

,CONCESIONARIO,IDLOCALID,NUM_PUNTOS_CRITICOS,TOTAL_OCURRENCIAS_SEMANALES
0,Bogotá Límpia (ASE04),10,153,167
1,Bogotá Límpia (ASE04),12,56,60
2,Ciudad Límpia (ASE03),8,71,138
3,Ciudad Límpia (ASE03),9,34,72
4,Lime (ASE02),6,14,16
5,Lime (ASE02),7,60,106
6,Lime (ASE02),13,14,18
7,Lime (ASE02),14,21,50
8,Lime (ASE02),15,19,21
9,Lime (ASE02),16,21,23


In [107]:
df_puntos_criticos['IDFRECUE'].nunique()

20

## 3. Exploración y análisis 